# Detection con ELIta
Questo notebook applica il lessico **ELIta** (originale e ricalcolato) ai commenti raccolti da r/Italia.

**Input**:
- `corpus_italy_estate.csv`: commenti con metadati
- `tokens_italy_estate.csv`: token con POS-tag
- `ELIta_INTENSITY_Matrix.csv`: matrice ELIta originale (Fase 1)
- Matrici ricalcolate con α = 0.2 / 0.5 / 0.8 (da `Valutazione_e_Ricalcolo.ipynb`)

**Output**:
- Tabella emozione / n.commenti / n.aggettivi (richiesta dalle specifiche)
- Confronto tra ELIta originale e versioni ricalcolate
- Grafici comparativi

## Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# --- Percorsi file ---
CORPUS_CSV   = "corpus_Italia_estate.csv"
TOKENS_CSV   = "tokens_Italia_estate.csv"
ELITA_CSV    = "../Fase1/ELIta_INTENSITY_Matrix.csv"
ALPHA_02_CSV = "../Fase2/output_csv/elita_recalculated_0_2.csv"
ALPHA_05_CSV = "../Fase2/output_csv/elita_recalculated_0_5.csv"
ALPHA_08_CSV = "../Fase2/output_csv/elita_recalculated_0_8.csv"
OUTPUT_DIR   = Path("output_confronto")
OUTPUT_DIR.mkdir(exist_ok=True)

from Fase1.emotion_config import BASIC_EMOTIONS, EMOTION_COLORS

# Caricamento corpus e token
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)

print(f"Commenti              : {len(df_corpus)}")
print(f"Token totali          : {len(df_tokens)}")

Commenti              : 622
Token totali          : 66867


## Caricamento e preparazione matrici ELIta

In [2]:
# Matrice originale — solo parole (no emoji)
df_matrix = pd.read_csv(ELITA_CSV, index_col=0)

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

df_elita_orig = df_matrix[df_matrix.index.map(is_not_emoji)][BASIC_EMOTIONS].fillna(0)
print(f"ELIta originale: {len(df_elita_orig)} parole")
df_elita_orig.head(5)

ELIta originale: 6719 parole


,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa
parola,,,,,,,,
????,0.12,0.42,0.42,0.17,0.17,0.04,0.79,0.17
a_caso,0.04,0.21,0.29,0.54,0.08,0.04,0.83,0.21
a_malincuore,0.00,0.83,0.21,0.21,0.21,0.00,0.04,0.12
a_scanso_di,0.17,0.25,0.42,0.17,0.04,0.25,0.21,0.58
abbagliante,0.46,0.04,0.12,0.12,0.00,0.17,0.71,0.62


In [3]:
df_elita_02 = pd.read_csv(ALPHA_02_CSV, index_col=0)
df_elita_05 = pd.read_csv(ALPHA_05_CSV, index_col=0)
df_elita_08 = pd.read_csv(ALPHA_08_CSV, index_col=0)

MATRICES = {
    "Originale (α=0)" : df_elita_orig,
    "Ibrido (α=0.2)"  : df_elita_02[BASIC_EMOTIONS].fillna(0),
    "Ibrido (α=0.5)"  : df_elita_05[BASIC_EMOTIONS].fillna(0),
    "Ibrido (α=0.8)"  : df_elita_08[BASIC_EMOTIONS].fillna(0),
}
print("\nMatrici pronte:", list(MATRICES.keys()))
MATRICES


Matrici pronte: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


{'Originale (α=0)':               gioia  tristezza  rabbia  paura  disgusto  fiducia  sorpresa  \
 parola                                                                       
 ????           0.12       0.42    0.42   0.17      0.17     0.04      0.79   
 a_caso         0.04       0.21    0.29   0.54      0.08     0.04      0.83   
 a_malincuore   0.00       0.83    0.21   0.21      0.21     0.00      0.04   
 a_scanso_di    0.17       0.25    0.42   0.17      0.04     0.25      0.21   
 abbagliante    0.46       0.04    0.12   0.12      0.00     0.17      0.71   
 ...             ...        ...     ...    ...       ...      ...       ...   
 zona           0.08       0.00    0.00   0.04      0.04     0.04      0.00   
 zotico         0.00       0.21    0.42   0.42      0.50     0.04      0.04   
 zucca          0.29       0.00    0.00   0.00      0.25     0.04      0.00   
 zucchero       0.71       0.17    0.00   0.04      0.04     0.29      0.04   
 zucchina       0.12       0.12  

## 3. Funzione di emotion detection
Per ogni commento:
1. Prendiamo i **lemmi degli aggettivi** (POS = ADJ) trovati da spaCy
2. Li cerchiamo nella matrice ELIta
3. Per ogni aggettivo trovato, prendiamo il vettore di intensità emotiva
4. Sommiamo i vettori → profilo emotivo del commento

In [4]:
def detect_emotions(df_corpus, df_tokens, df_elita):
    """
    Applica ELIta ai token di ogni commento (ADJ + NOUN + VERB).

    Restituisce:
        df_results : DataFrame con profilo emotivo per commento
        df_table   : tabella emozione / N.Commenti / N.Token
        unmatched  : lemmi non trovati in ELIta
    """
    # Filtra per ADJ, NOUN, VERB
    pos_filter = {"ADJ", "NOUN", "VERB"}
    df_filtered = df_tokens[df_tokens["pos"].isin(pos_filter)].copy()

    elita_index = set(df_elita.index)

    tokens_by_comment = df_filtered.groupby("comment_id")["lemma"].apply(list).to_dict()

    results = []
    unmatched_global = []

    for _, row in df_corpus.iterrows():
        cid   = row["comment_id"]
        lemmi = tokens_by_comment.get(cid, [])

        emotion_scores = {emo: 0.0 for emo in BASIC_EMOTIONS}
        found, not_found = [], []

        for lemma in lemmi:
            if lemma in elita_index:
                found.append(lemma)
                for emo in BASIC_EMOTIONS:
                    emotion_scores[emo] += df_elita.loc[lemma, emo]
            else:
                not_found.append(lemma)

        unmatched_global.extend(not_found)

        results.append({
            "comment_id"      : cid,
            "n_tokens_total"  : len(lemmi),
            "n_tokens_matched": len(found),
            "tokens_matched"  : found,
            **emotion_scores,
            "dominant_emotion": max(emotion_scores, key=emotion_scores.get)
                                 if any(v > 0 for v in emotion_scores.values()) else "neutrale"
        })

    df_results = pd.DataFrame(results)

    # Tabella riassuntiva richiesta dalle specifiche
    table_rows = []
    for emo in BASIC_EMOTIONS:
        comm_with_emo = df_results[df_results[emo] > 0]["comment_id"]
        n_tokens_emo  = df_filtered[
            df_filtered["comment_id"].isin(comm_with_emo) &
            df_filtered["lemma"].isin(elita_index)
        ]["lemma"].count()
        table_rows.append({
            "Emozione"  : emo.capitalize(),
            "N. Commenti": int((df_results[emo] > 0).sum()),
            "N. Token"   : int(n_tokens_emo)
        })
    df_table = pd.DataFrame(table_rows)

    return df_results, df_table, list(set(unmatched_global))


print("Funzione definita.")

Funzione definita.


## 4. Applicazione a tutte le versioni di ELIta

In [5]:
all_results = {}
all_tables  = {}

for version_name, df_elita in MATRICES.items():
    print(f"\nProcesso: {version_name}")
    df_res, df_tab, unmatched = detect_emotions(df_corpus, df_tokens, df_elita)
    all_results[version_name] = df_res
    all_tables[version_name]  = df_tab

    total   = df_res["n_tokens_total"].sum()
    matched = df_res["n_tokens_matched"].sum()
    rate    = matched / total * 100 if total > 0 else 0
    print(f"  Token trovati in ELIta       : {matched}/{total} ({rate:.1f}%)")
    print(f"  Commenti con almeno 1 match  : {(df_res['n_tokens_matched'] > 0).sum()}")

print("\nDone.")


Processo: Originale (α=0)
  Token trovati in ELIta       : 20837/27347 (76.2%)
  Commenti con almeno 1 match  : 621

Processo: Ibrido (α=0.2)
  Token trovati in ELIta       : 20837/27347 (76.2%)
  Commenti con almeno 1 match  : 621

Processo: Ibrido (α=0.5)
  Token trovati in ELIta       : 20837/27347 (76.2%)
  Commenti con almeno 1 match  : 621

Processo: Ibrido (α=0.8)
  Token trovati in ELIta       : 20837/27347 (76.2%)
  Commenti con almeno 1 match  : 621

Done.


## 5. Tabella riassuntiva (requisito 6 delle specifiche)

In [6]:
# Tabella per ELIta originale — quella richiesta dalle specifiche
print("=" * 50)
print("TABELLA EMOTION DETECTION — ELIta Originale")
print("=" * 50)
display(all_tables["Originale (α=0)"])

# Salviamo il CSV
all_tables["Originale (α=0)"].to_csv(OUTPUT_DIR / "emotion_table_originale.csv", index=False)
print("\nSalvata in 'output_confronto/emotion_table_originale.csv'")

TABELLA EMOTION DETECTION — ELIta Originale


,Emozione,N. Commenti,N. Token
0,Gioia,621,20837
1,Tristezza,621,20837
2,Rabbia,621,20837
3,Paura,620,20834
4,Disgusto,609,20809
5,Fiducia,621,20837
6,Sorpresa,621,20837
7,Aspettativa,621,20837



Salvata in 'output_confronto/emotion_table_originale.csv'


## 6. Confronto tra versioni ELIta

In [7]:
# Grafico: distribuzione emozione dominante per ogni versione
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(MATRICES.keys()),
    vertical_spacing=0.15
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for idx, (version_name, df_res) in enumerate(all_results.items()):
    row, col = positions[idx]
    counts = df_res["dominant_emotion"].value_counts()
    
    for emo in BASIC_EMOTIONS:
        n = counts.get(emo, 0)
        fig.add_trace(
            go.Bar(
                name=emo,
                x=[emo],
                y=[n],
                marker_color=EMOTION_COLORS[emo],
                showlegend=(idx == 0),
                legendgroup=emo
            ),
            row=row, col=col
        )

fig.update_layout(
    title_text="Distribuzione emozione dominante per versione ELIta",
    barmode="group",
    height=700
)
fig.show()

In [8]:
# Grafico: confronto punteggi medi per emozione tra le 4 versioni
means = {}
for version_name, df_res in all_results.items():
    means[version_name] = df_res[BASIC_EMOTIONS].mean()

df_means = pd.DataFrame(means).T.reset_index().rename(columns={"index": "Versione"})
df_melted = df_means.melt(id_vars="Versione", var_name="Emozione", value_name="Punteggio medio")

fig2 = px.bar(
    df_melted,
    x="Emozione", y="Punteggio medio",
    color="Versione",
    barmode="group",
    title="Punteggio emotivo medio per versione ELIta",
    color_discrete_sequence=["#455A64", "#1E88E5", "#FB8C00", "#E53935"]
)
fig2.show()

In [9]:
# Tabella confronto: quanti commenti cambiano emozione dominante tra originale e α=0.5?
orig  = all_results["Originale (α=0)"][["comment_id", "dominant_emotion"]].rename(columns={"dominant_emotion": "orig"})
alpha05 = all_results["Ibrido (α=0.5)"][["comment_id", "dominant_emotion"]].rename(columns={"dominant_emotion": "alpha05"})

df_compare = orig.merge(alpha05, on="comment_id")
cambiati = (df_compare["orig"] != df_compare["alpha05"]).sum()
totale   = len(df_compare)

print(f"Commenti che cambiano emozione dominante (Originale → α=0.5): {cambiati}/{totale} ({cambiati/totale*100:.1f}%)")
print()
print("Matrice di transizione (Originale → Ibrido α=0.5):")
pd.crosstab(df_compare["orig"], df_compare["alpha05"], margins=True)

Commenti che cambiano emozione dominante (Originale → α=0.5): 66/622 (10.6%)

Matrice di transizione (Originale → Ibrido α=0.5):


alpha05,aspettativa,disgusto,fiducia,gioia,neutrale,paura,rabbia,sorpresa,tristezza,All
orig,,,,,,,,,,
aspettativa,433,0,2,0,0,1,0,8,0,444
disgusto,0,2,0,0,0,0,0,0,0,2
fiducia,0,0,7,0,0,0,0,0,0,7
gioia,40,0,11,106,0,0,0,3,0,160
neutrale,0,0,0,0,1,0,0,0,0,1
paura,0,0,0,0,0,2,0,0,0,2
rabbia,0,0,0,0,0,0,2,0,0,2
sorpresa,0,0,0,0,0,0,0,2,0,2
tristezza,1,0,0,0,0,0,0,0,1,2


## 7. Coverage: token non trovati in ELIta

In [11]:
# Analisi dei token del corpus non coperti da ELIta
_, _, unmatched = detect_emotions(df_corpus, df_tokens, df_elita_orig)
unmatched_counts = pd.Series(unmatched).value_counts()

print(f"Lemmi non in ELIta: {len(unmatched_counts)}")
print(f"\nTop 30 lemmi mancanti:")
print(unmatched_counts.head(30).to_string())

# Salviamo per riferimento nella relazione
unmatched_counts.to_csv(OUTPUT_DIR / "token_non_coperti.csv", header=["frequenza"])
print("\nSalvati in 'output_confronto/token_non_coperti.csv'")

Lemmi non in ELIta: 4128

Top 30 lemmi mancanti:
anarchice               1
sessione                1
pub                     1
dormere                 1
sgomberare              1
calci                   1
vendere li              1
presentandome           1
tragicomico             1
fare le                 1
ingannevole             1
approvvigionamento      1
giustifichino           1
cronaca/20_luglio_01    1
sgradevola              1
radicato                1
beve                    1
lasciarmi               1
valre                   1
impesto                 1
figuriamoci             1
blackouts               1
restar                  1
bestiame                1
mementare               1
seppellito              1
ignorarere              1
aprendolo               1
casale                  1
scaricabile             1

Salvati in 'output_confronto/token_non_coperti.csv'


## 8. Salvataggio risultati completi

In [12]:
# Salviamo tutti i profili emotivi per ogni versione
for version_name, df_res in all_results.items():
    fname = "emotion_results_" + version_name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "") + ".csv"
    out_path = OUTPUT_DIR / fname
    df_res.to_csv(out_path, index=False)
    print(f"Salvato: {out_path}")

# Salviamo tutte le tabelle riassuntive in un unico file
df_all_tables = pd.concat(
    [tab.assign(Versione=vname) for vname, tab in all_tables.items()],
    ignore_index=True
)
df_all_tables.to_csv(OUTPUT_DIR / "emotion_tables_confronto.csv", index=False)
print("\nTutte le tabelle salvate in 'output_confronto/emotion_tables_confronto.csv'")
print(df_all_tables)

Salvato: output_confronto/emotion_results_Originale_α0.csv
Salvato: output_confronto/emotion_results_Ibrido_α0.2.csv
Salvato: output_confronto/emotion_results_Ibrido_α0.5.csv
Salvato: output_confronto/emotion_results_Ibrido_α0.8.csv

Tutte le tabelle salvate in 'output_confronto/emotion_tables_confronto.csv'
       Emozione  N. Commenti  N. Token         Versione
0         Gioia          621     20837  Originale (α=0)
1     Tristezza          621     20837  Originale (α=0)
2        Rabbia          621     20837  Originale (α=0)
3         Paura          620     20834  Originale (α=0)
4      Disgusto          609     20809  Originale (α=0)
5       Fiducia          621     20837  Originale (α=0)
6      Sorpresa          621     20837  Originale (α=0)
7   Aspettativa          621     20837  Originale (α=0)
8         Gioia          621     20837   Ibrido (α=0.2)
9     Tristezza          621     20837   Ibrido (α=0.2)
10       Rabbia          621     20837   Ibrido (α=0.2)
11        Paura   

---## Prossimo passo → Analisi qualitativaOra che hai i profili emotivi per ogni commento, il notebook successivo (`Fase4_AnalisiQualitativa.ipynb`) estrarrà i commenti più interessanti per l'analisi qualitativa (requisito E, vale +4 punti).